In [ ]:
import torch
import torch.nn as nn
from torchvision import models

class CNN(nn.Module):
    def __init__(self, dropout=0.3):
        super().__init__()

        weights = models.ResNet18_Weights.DEFAULT
        self.model = models.resnet18(weights=weights)

        num_features = self.model.fc.in_features

        self.model.fc = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(num_features, 10)
        )

    def forward(self, x):
        return self.model(x)

In [ ]:
import copy
import time
import matplotlib.pyplot as plt
import torch.optim as optim

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
print("Dispositivo:", device)

model = CNN(dropout=0.3).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(
    model.parameters(),
    lr=0.0001,
    weight_decay=0.0001
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2,
    min_lr=1e-7
)

max_epochs = 25
patience = 6
min_delta = 0.001

train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

best_val_loss = float("inf")
best_val_acc = 0.0
best_epoch = 0
best_weights = copy.deepcopy(model.state_dict())
epochs_without_improvement = 0

start_time = time.time()

In [ ]:
for epoch in range(max_epochs):

    # ENTRENAMIENTO
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = 100.0 * correct / total

    # VALIDACIÓN
    model.eval()
    val_running_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_running_loss += loss.item() * images.size(0)
            predictions = outputs.argmax(dim=1)

            val_correct += (predictions == labels).sum().item()
            val_total += labels.size(0)

    val_loss = val_running_loss / val_total
    val_acc = 100.0 * val_correct / val_total

    scheduler.step(val_loss)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_acc)
    val_accuracies.append(val_acc)

    current_lr = optimizer.param_groups[0]["lr"]

    print(
        f"Época {epoch + 1:02d}/{max_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Val Acc: {val_acc:.2f}% | "
        f"LR: {current_lr:.7f}"
    )

    # GUARDAR EL MEJOR MODELO SEGÚN VAL_LOSS
    if val_loss < best_val_loss - min_delta:
        best_val_loss = val_loss
        best_val_acc = val_acc
        best_epoch = epoch + 1
        best_weights = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0

        torch.save(best_weights, "mejor_modelo_gatos.pth")

    else:
        epochs_without_improvement += 1

    # EARLY STOPPING
    if epochs_without_improvement >= patience:
        print(f"Early stopping en la época {epoch + 1}")
        break

# RESTAURAR LOS MEJORES PESOS
model.load_state_dict(best_weights)

elapsed = (time.time() - start_time) / 60

print("\nEntrenamiento finalizado")
print("Mejor época:", best_epoch)
print(f"Mejor val_loss: {best_val_loss:.4f}")
print(f"Val_accuracy en esa época: {best_val_acc:.2f}%")
print(f"Tiempo de entrenamiento: {elapsed:.2f} minutos")

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Validation Loss")
plt.xlabel("Época")
plt.ylabel("Loss")
plt.title("Pérdida durante el entrenamiento")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(train_accuracies, label="Train Accuracy")
plt.plot(val_accuracies, label="Validation Accuracy")
plt.xlabel("Época")
plt.ylabel("Accuracy (%)")
plt.title("Precisión durante el entrenamiento")
plt.legend()
plt.grid(True)
plt.show()